# Training fits, validation chooses, testing judges
Use `finmlsim` for a dated return path, construct expanding validation folds, and reserve the final observations for one untouched assessment.

## 1. Using standard libraries
`TimeSeriesSplit` preserves order; we apply it only inside the development sample.

In [ ]:
import sys, subprocess
try:
    from finmlsim import simulate as sim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'finmlsim'])
    from finmlsim import simulate as sim
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

returns = sim.garch(n=160, seed=6)
X, y = returns[:-1, None], returns[1:]
development_end = len(X) - 30
splitter = TimeSeriesSplit(n_splits=3, test_size=25)
for fold, (train, validation) in enumerate(splitter.split(X[:development_end]), 1):
    print(f'fold {fold}: train {train[0]}–{train[-1]}, validate {validation[0]}–{validation[-1]}')
print(f'untouched test: {development_end}–{len(X)-1}')

## 2. Build the indices from scratch
The split is only a sequence of index boundaries; writing it out makes each job explicit.

In [ ]:
validation_size = 25
first_validation_end = development_end - 2 * validation_size
for fold in range(3):
    validation_end = first_validation_end + fold * validation_size
    validation_start = validation_end - validation_size
    train = list(range(validation_start))
    validation = list(range(validation_start, validation_end))
    print(f'fold {fold+1}: train {train[0]}–{train[-1]}, validate {validation[0]}–{validation[-1]}')
test = list(range(development_end, len(X)))
print(f'untouched test: {test[0]}–{test[-1]}')

## What to notice
The training window expands and validation always occurs later. The final 30 observations never select a model, threshold, or narrative.